# Z3-CSharp-09 : L'enigme d'Einstein (Zebra puzzle)

[← Z3-06 (optimisation avancee)](Z3-06-Advanced-Optimization-CSharp.ipynb) | [Jumeau Python](Z3-09-Enigme-Einstein-Python.ipynb) | [Serie Z3-API](README.md) | [Z3-10 (cryptarithmes) →](Z3-10-Cryptarithmetic-CSharp.ipynb)

## Le problème

L'**enigme d'Einstein** (ou *Zebra puzzle*) est un classique de logique : 5 maisons alignees, chacune habitee par une personne de nationalite, couleur, boisson, cigarette et animal différents. On dispose de **15 indices** (le Britannique vit dans la rouge, le Danois boit du the, la verte est a gauche de la blanche, etc.) et la question est : **qui possede le poisson ?**

A la main, la resolution procede par deductions progressives sur un tableau - c'est long et trompeur. L'espace de recherche brut est enorme : chaque attribut est une permutation des 5 maisons, soit $(5!)^5 = 24\,883\,200\,000$ combinaisons. Une **enumeration exhaustive** (brute force) testant les 15 indices pour chaque combinaison prend plusieurs minutes.

C'est typiquement la ou un solveur SMT comme Z3 **fait la différence** : on decrit les 25 variables (position de chaque valeur), le domaine, les 5 contraintes de permutation (`MkDistinct`) et les 15 indices, et le `Solver` trouve l'unique solution en **quelques millisecondes**. Le meme enonce se resout ici avec **Microsoft.Z3 natif**, sans couche declarative intermediaire.

> Jumeau C# du carnet Python [`Z3-09-Enigme-Einstein-Python.ipynb`](Z3-09-Enigme-Einstein-Python.ipynb), qui porte lui-meme l'exemple [`18_Einsteins_Riddle`](../Z3-Linq2Z3/18_Einsteins_Riddle.ipynb) de la serie sœur Z3.Linq. Politique de jumelage : #26070.

In [1]:
#r "nuget: Microsoft.Z3"
#load "Z3NativeLoader.cs"
using Microsoft.Z3;
using System.Diagnostics;

// Bibliotheque native de Z3 hors Windows et macOS Intel (voir Z3NativeLoader.cs)
Z3NativeLoader.Register(typeof(Context).Assembly);
Console.WriteLine("Imports OK : Microsoft.Z3 version " + Microsoft.Z3.Version.FullVersion);


The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Microsoft.Z3, 4.12.2

Imports OK : Microsoft.Z3 version Z3 4.12.2.0


## Modelisation : l'encodage par position

L'astuce centrale consiste a **inverser le point de vue**. Au lieu de demander « quelle couleur a la maison *k* ? », on définit pour chaque valeur d'attribut une variable entiere egale a la **position** (numéro de maison 0..4) ou elle se trouve :

- `Brit`, `Swede`, `Dane`, `Norwegian`, `German` : position de chaque nationalite
- `Red`, `Green`, `White`, `Yellow`, `Blue` : position de chaque couleur
- `Tea`, `Coffee`, `Milk`, `Water`, `Beer` : position de chaque boisson
- `PallMall`, `Dunhill`, `Blend`, `BlueMaster`, `Prince` : position de chaque cigarette
- `Dog`, `Bird`, `Cat`, `Horse`, `Fish` : position de chaque animal

Soit **25 variables entieres** (`IntExpr`). Les contraintes se traduisent alors très naturellement :

1. **Domaine** : chaque variable $\in \{0,1,2,3,4\}$.
2. **Permutation** : dans chaque groupe d'attributs, les 5 positions sont deux a deux distinctes (`MkDistinct`) - chaque valeur occupe une maison différente.
3. **Les 15 indices** : des egalites de position (`Brit == Red` : le Britannique et la rouge sont dans la même maison) ou des **adjacences** (`|Blend - Cat| == 1` : le fumeur de Blend est voisin du Chat).

Les 15 indices sont portes par une table **nommee** : le modele se recompose ensuite par sous-ensemble, ce dont les exercices se servent.


In [2]:
static class Einstein
{
    // Un seul Context pour tout le carnet : deux Context differents ne melangent pas leurs expressions.
    public static readonly Context Ctx = new Context();

    // 25 variables de position (0..4) : une par valeur d'attribut.
    public static readonly (string Groupe, string[] Valeurs)[] Groups =
    {
        ("Nationalite", new[] { "Brit", "Swede", "Dane", "Norwegian", "German" }),
        ("Couleur",     new[] { "Red", "Green", "White", "Yellow", "Blue" }),
        ("Boisson",     new[] { "Tea", "Coffee", "Milk", "Water", "Beer" }),
        ("Cigarette",   new[] { "PallMall", "Dunhill", "Blend", "BlueMaster", "Prince" }),
        ("Animal",      new[] { "Dog", "Bird", "Cat", "Horse", "Fish" }),
    };

    // Les 15 indices, nommes de "1" a "15".
    public static readonly (string Nom, Func<Dictionary<string, IntExpr>, BoolExpr> Expr)[] Indices =
    {
        ("1",  v => Ctx.MkEq(v["Brit"], v["Red"])),                                              // le Britannique vit dans la rouge
        ("2",  v => Ctx.MkEq(v["Swede"], v["Dog"])),                                             // le Suedois eleve des chiens
        ("3",  v => Ctx.MkEq(v["Dane"], v["Tea"])),                                              // le Danois boit du the
        ("4",  v => Ctx.MkEq(v["Green"], Ctx.MkSub(v["White"], Ctx.MkInt(1)))),                  // la verte est juste a gauche de la blanche
        ("5",  v => Ctx.MkEq(v["Green"], v["Coffee"])),                                          // la verte boit du cafe
        ("6",  v => Ctx.MkEq(v["PallMall"], v["Bird"])),                                         // Pall Mall eleve des oiseaux
        ("7",  v => Ctx.MkEq(v["Yellow"], v["Dunhill"])),                                        // la jaune fume du Dunhill
        ("8",  v => Ctx.MkEq(v["Milk"], Ctx.MkInt(2))),                                          // la maison du milieu boit du lait
        ("9",  v => Ctx.MkEq(v["Norwegian"], Ctx.MkInt(0))),                                     // le Norvegien dans la 1ere maison
        ("10", v => Voisin(v, "Blend", "Cat")),                                                  // Blend voisin du Chat
        ("11", v => Voisin(v, "Horse", "Dunhill")),                                              // Cheval voisin du Dunhill
        ("12", v => Ctx.MkEq(v["BlueMaster"], v["Beer"])),                                       // BlueMaster boit de la biere
        ("13", v => Ctx.MkEq(v["German"], v["Prince"])),                                         // l'Allemand fume du Prince
        ("14", v => Voisin(v, "Norwegian", "Blue")),                                             // Norvegien voisin de la bleue
        ("15", v => Voisin(v, "Blend", "Water")),                                                // Blend voisin de l'Eau
    };

    static BoolExpr Voisin(Dictionary<string, IntExpr> v, string a, string b) =>
        Ctx.MkOr(
            Ctx.MkEq(Ctx.MkSub(v[a], v[b]), Ctx.MkInt(1)),
            Ctx.MkEq(Ctx.MkSub(v[b], v[a]), Ctx.MkInt(1)));

    /// Construit le solveur : domaine 0..4, un MkDistinct par groupe, puis les indices
    /// dont le nom est liste (tous quand <paramref name="noms"/> est null).
    public static (Solver S, Dictionary<string, IntExpr> V) Build(IEnumerable<string> noms = null)
    {
        var s = Ctx.MkSolver();
        var v = Groups
            .SelectMany(g => g.Valeurs)
            .ToDictionary(n => n, n => (IntExpr)Ctx.MkIntConst(n));

        // (1) Domaine 0..4
        foreach (var x in v.Values)
            s.Add(Ctx.MkAnd(Ctx.MkGe(x, Ctx.MkInt(0)), Ctx.MkLe(x, Ctx.MkInt(4))));

        // (2) Permutation : les 5 valeurs d'un meme attribut occupent des maisons distinctes
        foreach (var g in Groups)
            s.Add(Ctx.MkDistinct(g.Valeurs.Select(n => (Expr)v[n]).ToArray()));

        // (3) Les indices demandes (tous par defaut)
        var gardes = noms == null ? null : new HashSet<string>(noms);
        foreach (var (nom, expr) in Indices)
            if (gardes == null || gardes.Contains(nom))
                s.Add(expr(v));

        return (s, v);
    }
}

Console.WriteLine("Modele encode : 25 variables de position, 5 groupes Distinct, 15 indices.");


Modele encode : 25 variables de position, 5 groupes Distinct, 15 indices.


## Resolution et temoin

On appelle `Check()` : si le statut est `SATISFIABLE`, on extrait le modèle puis on **inverse** l'encodage par position - pour chaque maison $h \in \{0..4\}$, on cherche quelle valeur de chaque attribut y se trouve (la valeur dont la position vaut $h$). Cela reconstitue le tableau classique de la solution.


In [3]:
// Labels francais pour l'affichage (associes au nom de variable).
var LabelsFr = new (string Groupe, (string Fr, string Cle)[] Valeurs)[]
{
    ("Nationalite", new[] { ("Britannique", "Brit"), ("Suedois", "Swede"), ("Danois", "Dane"), ("Norvegien", "Norwegian"), ("Allemand", "German") }),
    ("Couleur",     new[] { ("Rouge", "Red"), ("Verte", "Green"), ("Blanche", "White"), ("Jaune", "Yellow"), ("Bleue", "Blue") }),
    ("Boisson",     new[] { ("The", "Tea"), ("Cafe", "Coffee"), ("Lait", "Milk"), ("Eau", "Water"), ("Biere", "Beer") }),
    ("Cigarette",   new[] { ("Pall Mall", "PallMall"), ("Dunhill", "Dunhill"), ("Blend", "Blend"), ("BlueMaster", "BlueMaster"), ("Prince", "Prince") }),
    ("Animal",      new[] { ("Chien", "Dog"), ("Oiseau", "Bird"), ("Chat", "Cat"), ("Cheval", "Horse"), ("Poisson", "Fish") }),
};

// Resout l'enigme et affiche le tableau des 5 maisons.
var (S9, V9) = Einstein.Build();
if (S9.Check() != Status.SATISFIABLE)
    throw new Exception("UNSAT (inattendu pour cette enigme)");
var M9 = S9.Model;
var pos = V9.ToDictionary(kv => kv.Key, kv => ((IntNum)M9.Evaluate(kv.Value)).Int64);

string ValAt(string groupe, long h)
{
    var valeurs = LabelsFr.First(g => g.Groupe == groupe).Valeurs;
    foreach (var (fr, cle) in valeurs)
        if (pos[cle] == h)
            return fr;
    return "?";
}

var cols = new[] { "Maison", "Nationalite", "Couleur", "Boisson", "Cigarette", "Animal" };
var widths = new[] { 7, 13, 9, 9, 13, 9 };
Console.WriteLine(string.Join("  ", cols.Select((c, i) => c.PadRight(widths[i]))));
Console.WriteLine("  " + new string('-', widths.Sum() + 2 * (cols.Length - 1)));
for (int h = 0; h < 5; h++)
{
    var row = new[] { (h + 1).ToString(), ValAt("Nationalite", h), ValAt("Couleur", h), ValAt("Boisson", h), ValAt("Cigarette", h), ValAt("Animal", h) };
    Console.WriteLine(string.Join("  ", row.Select((c, i) => c.PadRight(widths[i]))));
}
Console.WriteLine();
Console.WriteLine($">>> Reponse : c'est le {ValAt("Nationalite", pos["Fish"])} qui possede le Poisson (maison {pos["Fish"] + 1}).");


Maison   Nationalite    Couleur    Boisson    Cigarette      Animal   


  ----------------------------------------------------------------------


1        Norvegien      Jaune      Eau        Dunhill        Chat     


2        Danois         Bleue      The        Blend          Cheval   


3        Britannique    Rouge      Lait       Pall Mall      Oiseau   


4        Allemand       Verte      Cafe       Prince         Poisson  


5        Suedois        Blanche    Biere      BlueMaster     Chien    


>>> Reponse : c'est le Allemand qui possede le Poisson (maison 4).


## Interpretation

Z3 trouve l'unique solution en quelques millisecondes. La resolution humaine procederait par deductions progressives - on place le Norvegien en 1 (`Norwegian == 0`), le lait en 3 (`Milk == 2`), puis on propage les egalites et adjacences - mais c'est précisément ce travail de **propagation de contraintes** que le solveur automatise, sans erreur ni oubli.

La reponse a la question « qui possede le poisson ? » tombe directement du modèle extrait. Notons qu'**aucun indice ne parle du poisson** : c'est par elimination (les 4 autres animaux sont localises par les indices) que sa position se deduit.


## Vérification du témoin : relire les indices

Un modèle extrait n'est pas un résultat à prendre sur parole — vérifions que le tableau ci-dessus satisfait bien quelques indices choisis dans l'énoncé original :

- **Le Norvégien vit à côté de la maison bleue** : Norvégien en maison 1, Bleue en maison 2 — voisins.
- **Le fumeur de Blend a un voisin qui garde le chat** : Blend en maison 2, Chat en maison 1 — voisins.
- **La maison verte est immédiatement à gauche de la blanche** : Verte en 4, Blanche en 5.
- **La maison du centre boit du lait** : Lait en maison 3.

Ce geste de re-lecture croisée est l'équivalent d'un test unitaire sur le solveur : il valide l'**encodage** (une contrainte mal traduite produirait un témoin qui échoue à la vérification), pas seulement la satisfiabilité.


## Approche naive vs solveur : mesurer l'ecart

Plutot que d'affirmer que le solveur est plus rapide, **mesurons** l'ecart. L'espace de recherche brut est $(5!)^5$ : chaque attribut est une permutation des 5 maisons, independamment. On chronometre aussi la resolution Z3, puis on **prouve l'unicite** de la solution en ajoutant la negation du temoin trouve et en re-resolvant (on attend `UNSATISFIABLE`).


In [4]:
// (1) Espace de recherche brut
long fact5 = 1;
for (int i = 2; i <= 5; i++) fact5 *= i;
long space = 1;
for (int i = 0; i < 5; i++) space *= fact5;   // (5!)^5 = 24 883 200 000
double bruteSeconds = space / 1e8;            // a ~1e8 tests/s, sans compter le test des 15 indices

// (2) Chronometre la resolution Z3
var chrono = Stopwatch.StartNew();
var (S1, V1) = Einstein.Build();
if (S1.Check() != Status.SATISFIABLE)
    throw new Exception("UNSAT inattendu");
var M1 = S1.Model;
chrono.Stop();
double z3Ms = chrono.Elapsed.TotalMilliseconds;

// (3) Unicite : on nie le temoin trouve (au moins une variable differe) -> on attend UNSAT
var (S2, V2) = Einstein.Build();
var temoin = V2.ToDictionary(kv => kv.Key, kv => ((IntNum)M1.Evaluate(kv.Value)).Int64);
S2.Add(Einstein.Ctx.MkOr(V2.Select(kv =>
    Einstein.Ctx.MkNot(Einstein.Ctx.MkEq(kv.Value, Einstein.Ctx.MkInt(temoin[kv.Key])))).ToArray()));
bool unique = S2.Check() == Status.UNSATISFIABLE;

Console.WriteLine($"Espace de recherche brut       : (5!)^5 = {space:N0} combinaisons");
Console.WriteLine($"Brute force estime (~1e8/s)    : {bruteSeconds:F1} s (hors test des 15 indices)");
Console.WriteLine($"Z3 solve temps                 : {z3Ms:F1} ms");
Console.WriteLine($"Solution unique (UNSAT si nie) : {unique}");
Console.WriteLine($" -> Z3 resout en millisecondes un espace de ~{space / 1e9:F0} milliards, et prouve l'unicite sans enumeration.");


Espace de recherche brut       : (5!)^5 = 24 883 200 000 combinaisons


Brute force estime (~1e8/s)    : 248,8 s (hors test des 15 indices)


Z3 solve temps                 : 29,0 ms


Solution unique (UNSAT si nie) : True


 -> Z3 resout en millisecondes un espace de ~25 milliards, et prouve l'unicite sans enumeration.


## Lecture du benchmark

Sur cette enigme, l'ecart est **frappant** : Z3 resout les 25 variables entrelacees en quelques millisecondes, alors que la brute force doit affronter **~25 milliards** de combinaisons (soit plusieurs minutes même a cadence elevee, et bien plus avec le test des 15 indices a chaque candidat).

La **propagation** fait tout le travail : `Norwegian == 0` et `Milk == 2` sont des ancres imposees par les indices 9 et 8, puis les egalites de position et les adjacences reduisent les domaines restants jusqu'a ce qu'il ne reste qu'une seule combinaison coherente. C'est exactement la mecanique qu'un humain deployerait a la main sur le tableau - mais le solveur la realise sans erreur et la **prouve complete** (unicite via `UNSATISFIABLE` sur la negation).


## Exercices

Les trois exercices initiaux de ce carnet ont été résolus par un rendu de TP : chacun devient un **exemple guidé** (code de l'étudiante conservé tel quel, porté en C#), suivi d'une lecture du résultat puis d'un **nouvel exercice** non résolu, de difficulté comparable et portant le même concept Z3. Vous y réutiliserez l'encodage par position et la fonction `Einstein.Build()`.


### Exemple guidé 1 -- Un 16e indice : le Chat boit de l'Eau

Résolu par @EmmaLehec (TP2 IA2-Min2, groupe G02, PR #27026) dans le carnet Python, intégré ici comme exemple guidé.

**Objectif** : ajouter l'indice `Cat == Water` (le Chat et l'Eau sont dans la même maison) et observer si l'énigme reste satisfaisable (`sat`) ou devient contradictoire (`unsat`).

**Rappel** : on reprend `Einstein.Build()`, on ajoute `s.Add(Einstein.Ctx.MkEq(v["Cat"], v["Water"]))` avant `Check()`, et on compare avec la solution trouvée plus haut — le Chat et l'Eau y sont-ils dans la même maison ?


In [5]:
// Exemple guide 1 : ajout de l'indice Cat == Water et test de la satisfiabilite.
string TesterChatEau()
{
    // Etape 1 : le solveur deja configure avec les 15 indices
    var (s, v) = Einstein.Build();

    // Etape 2 : le 16e indice (le Chat et l'Eau dans la meme maison)
    s.Add(Einstein.Ctx.MkEq(v["Cat"], v["Water"]));

    // Etape 3 : le probleme reste-t-il possible ? ("sat" / "unsat", comme le jumeau Python)
    return s.Check() == Status.SATISFIABLE ? "sat" : "unsat";
}

Console.WriteLine("Verdict avec Cat == Water : " + TesterChatEau());


Verdict avec Cat == Water : sat


### Lecture du résultat : un indice déjà vrai

Le verdict est `sat` : l'énigme reste satisfaisable. Ce n'est pas un hasard — dans l'unique solution affichée plus haut, le Chat et l'Eau sont **déjà** dans la même maison (maison 1, chez le Norvégien). Le 16e indice est donc déjà vrai : il ne crée aucune contradiction et n'apporte aucune information nouvelle. Un indice qui contredirait la solution (par exemple `Fish == Water`) rendrait au contraire l'énigme `unsat` — c'est exactement l'objet du nouvel exercice 1.


### Exercice 1 -- Prédire le verdict d'un 17e indice

**Objectif** : généraliser l'exemple guidé 1 en une fonction qui classe **n'importe quel** indice candidat, puis répondre à la question conceptuelle.

Écrire `ClasserIndice(indice)` qui prend un indice candidat — un `Func<Dictionary<string, IntExpr>, BoolExpr>`, sur le patron des lambdas de `Einstein.Indices` — reconstruit le solveur complet, ajoute l'indice et retourne `"sat"` ou `"unsat"`. Classer ensuite les quatre candidats :

1. `v => ctx.MkEq(v["Cat"], v["Water"])` — le Chat et l'Eau dans la même maison ;
2. `v => ctx.MkEq(v["Fish"], v["Water"])` — le Poisson et l'Eau dans la même maison ;
3. `v => ctx.MkEq(v["Horse"], v["Blend"])` — le Cheval dans la maison du fumeur de Blend ;
4. `v => ctx.MkEq(v["Bird"], v["Milk"])` — l'Oiseau dans la maison au lait.

**Question** : parmi les verdicts obtenus, peut-il exister un indice qui donne `sat` tout en **changeant** la solution du tableau ? Justifier à partir de l'unicité prouvée plus haut.

**Indices** : chaque appel doit repartir d'un solveur **frais** (`Einstein.Build()`) — sinon les indices s'accumulent d'un appel à l'autre. Pour la question : que vaut l'ensemble des solutions avant l'ajout, et que lui fait un indice déjà vrai ?


In [6]:
// Exercice 1 : classer un 17e indice candidat (sat / unsat).
string ClasserIndice(Func<Dictionary<string, IntExpr>, BoolExpr> indice)
{
    // TODO etudiant : Einstein.Build(), s.Add(indice(v)), puis retourner "sat" ou "unsat"
    return "a completer"; // TODO etudiant
}

// Les 4 indices candidats de l'enonce.
var candidats = new (string Nom, Func<Dictionary<string, IntExpr>, BoolExpr> Indice)[]
{
    ("Cat == Water",  v => Einstein.Ctx.MkEq(v["Cat"], v["Water"])),
    ("Fish == Water", v => Einstein.Ctx.MkEq(v["Fish"], v["Water"])),
    ("Horse == Blend", v => Einstein.Ctx.MkEq(v["Horse"], v["Blend"])),
    ("Bird == Milk",  v => Einstein.Ctx.MkEq(v["Bird"], v["Milk"])),
};
foreach (var (nom, indice) in candidats)
    Console.WriteLine($"{nom} : {ClasserIndice(indice)}");

Console.WriteLine("Exercice a completer : classer les 4 indices candidats.");


Cat == Water : a completer


Fish == Water : a completer


Horse == Blend : a completer


Bird == Milk : a completer


Exercice a completer : classer les 4 indices candidats.


### Exemple guidé 2 -- Énumérer toutes les solutions par blocage itératif

Résolu par @EmmaLehec (TP2 IA2-Min2, groupe G02, PR #27026) dans le carnet Python, intégré ici comme exemple guidé.

**Objectif** : énumérer **toutes** les solutions de l'énigme par **blocage itératif** (on résout, on compte, on ajoute une contrainte interdisant le témoin trouvé, on re-résout) jusqu'à `UNSATISFIABLE`. Le compte attendu est 1 (solution unique), ce qui confirme le benchmark ci-dessus. Le rendu généralise la fonction en `CompterSolutions(s, v, limite)` pour qu'elle s'applique à n'importe quel solveur — elle resservira dans l'exemple guidé 3 et le nouvel exercice 2.


In [7]:
// Exemple guide 2 : enumerer toutes les solutions par blocage iteratif.
long CompterSolutions(Solver s = null, Dictionary<string, IntExpr> v = null, long limite = 1000)
{
    // Etape 1 : le solveur avec les 15 indices
    if (s == null)
        (s, v) = Einstein.Build();

    // Etape 2 : boucle de blocage iteratif
    long compte = 0;
    while (compte < limite && s.Check() == Status.SATISFIABLE)
    {
        compte++;
        var m = s.Model;   // la solution trouvee
        s.Add(Einstein.Ctx.MkOr(v.Select(kv => Einstein.Ctx.MkNot(
            Einstein.Ctx.MkEq(kv.Value, Einstein.Ctx.MkInt(((IntNum)m.Evaluate(kv.Value)).Int64)))).ToArray()));
    }
    return compte;
}

long nb = CompterSolutions();
Console.WriteLine($"Nombre de solutions de l'enigme d'Einstein : {nb} (attendu : 1)");


Nombre de solutions de l'enigme d'Einstein : 1 (attendu : 1)


In [8]:
// Bonus exemple guide 2 : la boucle compte-t-elle vraiment ? On retire un indice et on recompte.
// Le Python retire un indice en filtrant les assertions une a une ; ici la table nommee
// fait le meme travail : on reconstruit le solveur sans le nom vise.
var tests = new (string Nom, string Description)[]
{
    ("9",  "9  (Norvegien en maison 1)"),
    ("15", "15 (Blend voisin de l'Eau)"),
};
foreach (var (nom, description) in tests)
{
    var (s2, v2) = Einstein.Build(Einstein.Indices.Select(i => i.Nom).Where(n => n != nom));
    Console.WriteLine($"Sans l'indice {description} : {CompterSolutions(s2, v2)} solution(s)");
}


Sans l'indice 9  (Norvegien en maison 1) : 17 solution(s)


Sans l'indice 15 (Blend voisin de l'Eau) : 1 solution(s)


### Lecture du résultat : la boucle compte vraiment, et un indice est redondant

Le blocage itératif (trouver une solution, l'interdire par un `MkOr` sur les 25 variables, recommencer jusqu'à `UNSATISFIABLE`) donne **1 solution** : l'énigme d'Einstein est bien unique, ce qui confirme le benchmark. Pour vérifier que la boucle compte réellement, on retire un indice et on recompte : sans l'indice 9 (Norvégien en maison 1), on trouve **17 solutions**. Sans l'indice 15 (Blend voisin de l'Eau), il en reste **1** : cet indice est **redondant**, les 14 autres suffisent à déterminer la solution. Le nouvel exercice 2 pousse cette question un cran plus loin.


### Exercice 2 -- Un sous-ensemble minimal d'indices

**Objectif** : sans l'indice 15, l'énigme garde exactement 1 solution — mais est-il le seul qu'on puisse retirer ? Trouver par **élimination gloutonne** un sous-ensemble des 15 indices qui garde **exactement 1 solution** : partir de la liste complète, et pour chaque indice dans l'ordre, essayer de le retirer **définitivement** — si le reste a toujours exactement 1 solution, il reste retiré, sinon il est remis. Combien d'indices le résultat conserve-t-il, et lesquels ?

**Indices** : `Einstein.Indices` donne les 15 indices nommés, et `Einstein.Build(noms)` reconstruit un solveur frais ne contenant que les noms passés. Comptez avec `CompterSolutions(s, v)`. Affichez le nombre d'indices conservés et leur liste. Attention : un indice isolément redondant peut devenir nécessaire une fois d'autres indices retirés — d'où le test à chaque étape.


In [9]:
// Exercice 2 : un sous-ensemble minimal d'indices (elimination gloutonne).
List<string> IndicesMinimaux()
{
    // TODO etudiant : partir de tous les noms, essayer de retirer chaque indice, et garder
    // le retrait si Einstein.Build(noms).CompterSolutions garde exactement 1 solution.
    return null; // TODO etudiant
}

Console.WriteLine("Exercice a completer : elimination gloutonne des indices redondants.");


Exercice a completer : elimination gloutonne des indices redondants.


### Exemple guidé 3 -- Mini-Zebra à 4 maisons

Résolu par @EmmaLehec (TP2 IA2-Min2, groupe G02, PR #27026) dans le carnet Python, intégré ici comme exemple guidé.

**Objectif** : modéliser une **mini-énigme** à 4 maisons et 3 attributs (4 nationalités, 4 couleurs, 4 animaux) avec 8 indices, et vérifier que l'encodage par position se généralise — le modèle est paramètre par les données, seule l'instance change.


In [10]:
// Exemple guide 3 : un mini-Zebra a 4 maisons / 3 attributs.
(Solver S, Dictionary<string, IntExpr> V) ResoudreMiniZebra()
{
    var ctx = Einstein.Ctx;
    var groups = new (string Groupe, string[] Valeurs)[]
    {
        ("Nationalite", new[] { "Francais", "Anglais", "Espagnol", "Italien" }),
        ("Couleur",     new[] { "Rouge", "Bleu", "Vert", "Jaune" }),
        ("Animal",      new[] { "Chien", "Chat", "Poisson", "Oiseau" }),
    };

    var s = ctx.MkSolver();
    var v = groups.SelectMany(g => g.Valeurs).ToDictionary(n => n, n => (IntExpr)ctx.MkIntConst(n));

    // Domaine 0..3 pour les 12 variables
    foreach (var x in v.Values)
        s.Add(ctx.MkAnd(ctx.MkGe(x, ctx.MkInt(0)), ctx.MkLe(x, ctx.MkInt(3))));

    // Les 4 valeurs de chaque attribut occupent des maisons distinctes
    foreach (var g in groups)
        s.Add(ctx.MkDistinct(g.Valeurs.Select(n => (Expr)v[n]).ToArray()));

    // Les 8 indices
    s.Add(ctx.MkEq(v["Francais"], v["Rouge"]));                                            // 1. le Francais habite la rouge
    s.Add(ctx.MkEq(v["Anglais"], v["Chien"]));                                             // 2. l'Anglais possede le chien
    s.Add(ctx.MkEq(v["Espagnol"], v["Vert"]));                                             // 3. l'Espagnol habite la verte
    s.Add(ctx.MkEq(v["Francais"], v["Chat"]));                                             // 4. le Francais possede le chat
    s.Add(ctx.MkEq(v["Espagnol"], ctx.MkInt(0)));                                          // 5. l'Espagnol en premiere maison
    s.Add(ctx.MkOr(ctx.MkEq(ctx.MkSub(v["Chien"], v["Jaune"]), ctx.MkInt(1)),
                   ctx.MkEq(ctx.MkSub(v["Jaune"], v["Chien"]), ctx.MkInt(1))));            // 6. le chien voisin de la jaune
    s.Add(ctx.MkEq(v["Poisson"], v["Vert"]));                                              // 7. le poisson dans la verte
    s.Add(ctx.MkOr(ctx.MkEq(ctx.MkSub(v["Bleu"], v["Vert"]), ctx.MkInt(1)),
                   ctx.MkEq(ctx.MkSub(v["Vert"], v["Bleu"]), ctx.MkInt(1))));              // 8. la bleue voisine de la verte

    return (s, v);
}

var (Smz, Vmz) = ResoudreMiniZebra();
if (Smz.Check() == Status.SATISFIABLE)
{
    Console.WriteLine("Solution trouvee !");
    var mmz = Smz.Model;
    var attributs = new (string Groupe, string[] Valeurs)[]
    {
        ("Nationalite", new[] { "Francais", "Anglais", "Espagnol", "Italien" }),
        ("Couleur",     new[] { "Rouge", "Bleu", "Vert", "Jaune" }),
        ("Animal",      new[] { "Chien", "Chat", "Poisson", "Oiseau" }),
    };
    for (int maison = 0; maison < 4; maison++)
    {
        Console.WriteLine($"\nMaison {maison + 1} :");
        foreach (var (groupe, valeurs) in attributs)
            foreach (var valeur in valeurs)
                if (((IntNum)mmz.Evaluate(Vmz[valeur])).Int64 == maison)
                    Console.WriteLine($"  {groupe} : {valeur}");
    }
}
else
{
    Console.WriteLine("Aucune solution.");
}

// Etape 3 : solution unique ? On reutilise CompterSolutions (exemple guide 2).
var (Smz2, Vmz2) = ResoudreMiniZebra();
Console.WriteLine($"\nNombre de solutions : {CompterSolutions(Smz2, Vmz2)} (attendu : 1)");


Solution trouvee !



Maison 1 :


  Nationalite : Espagnol


  Couleur : Vert


  Animal : Poisson



Maison 2 :


... (8 outputs left out)

  Nationalite : Francais


  Couleur : Rouge


  Animal : Chat



Nombre de solutions : 1 (attendu : 1)


### Lecture du résultat : l'encodage se généralise tel quel

Le même encodage par position passe directement à 4 maisons : domaine `0..3`, 3 groupes `MkDistinct` et 8 indices (égalités de position et adjacences). Trouver une solution (`sat`) ne suffit pas pour une énigme : il faut aussi qu'elle soit **unique**. En réutilisant `CompterSolutions` (exemple guidé 2), on vérifie que les 8 indices déterminent **une seule solution** : l'Espagnol, dans la maison verte, possède le poisson.


### Exercice 3 -- Cinq amies, cinq boissons, cinq loisirs

**Objectif** : modéliser une énigme à **5 maisons** sur un autre thème, avec deux types de relations **nouveaux** par rapport au mini-Zebra : « à gauche quelque part » (`a < b`, soit `ctx.MkLt`) et « immédiatement à gauche de » (`a == b - 1`). Cinq amies (Alice, Bruno, Chloe, David, Emilie), cinq boissons (Cafe, The, Jus, Lait, Eau) et cinq loisirs (Echecs, Lecture, Course, Peinture, Musique) — une valeur par maison, une maison par valeur.

Les 11 indices :

1. Alice habite la première maison.
2. Emilie habite la dernière maison.
3. Qui boit du lait fait de la peinture.
4. Bruno boit du thé.
5. Le joueur d'échecs habite quelque part à gauche du buveur de café.
6. David boit du jus.
7. Le coureur habite immédiatement à gauche de Chloe.
8. La lectrice boit de l'eau.
9. Chloe fait de la musique.
10. Le buveur de café habite immédiatement à droite de Bruno.
11. La peinture se pratique dans une maison à gauche de celle de Chloe.

**Indices** : reprenez la structure de `ResoudreMiniZebra` (domaine `0..4`, 3 groupes `MkDistinct`) ; « immédiatement à gauche de » se code `a == b - 1` et « à gauche quelque part » `ctx.MkLt(a, b)`. Vérifiez `sat` **puis** l'unicité avec `CompterSolutions(s, v)` — attendu : exactement 1 solution.


In [11]:
// Exercice 3 : cinq amies, cinq boissons, cinq loisirs (5 maisons, 3 attributs, 11 indices).
(Solver S, Dictionary<string, IntExpr> V) ResoudreCinqAmies()
{
    // Etape 1 : groups_5 (Prenom, Boisson, Loisir), domaine 0..4 + 3 MkDistinct.
    // Etape 2 : les 11 indices de l'enonce (egalites, adjacences, "a gauche", "immediatement a gauche").
    // Etape 3 : Check() -> sat, puis CompterSolutions(s, v) -> attendu exactement 1.
    // TODO etudiant
    return (null, null); // TODO etudiant
}

Console.WriteLine("Exercice a completer : l'enigme des cinq amies (5 maisons, 11 indices).");


Exercice a completer : l'enigme des cinq amies (5 maisons, 11 indices).


## Conclusion

L'enigme d'Einstein illustre le gain du **declaratif** sur une classe de problemes ou l'approche humaine (deduction progressive sur un tableau) et l'approche naive (brute force) sont toutes deux laborieuses :

- L'**encodage par position** (une variable entiere = position de chaque valeur) transforme un puzzle apparemment qualitatif en un système de contraintes arithmetiques : egalites (`Brit == Red`), adjacences (`|Blend - Cat| == 1`), permutations (`MkDistinct`).
- La **propagation de contraintes** resout les 25 variables entrelacees en millisecondes la ou la brute force affronte ~25 milliards de combinaisons ; le solveur **prouve l'unicite** en niant le temoin et en observant `UNSATISFIABLE`.
- Le modèle est **paramètre par les données** : ajouter un indice, un attribut ou une maison se fait en modifiant l'instance, sans re-ecrire l'algorithme de resolution.

**Contraste avec le job-shop (carnet 08)** : ici `Solver` suffit (on cherche une solution realisable, pas un optimum) - c'est la **satisfiabilite** pure, la ou l'ordonnancement relevait de l'**optimisation** (`Optimize.minimize` du makespan). Les deux postures du solveur, posees au carnet 01, trouvent ici leur illustration complementaire.

Suite logique : le jumeau Python [`Z3-09-Enigme-Einstein-Python.ipynb`](Z3-09-Enigme-Einstein-Python.ipynb) pour la version la plus courte du meme modele, ou la [serie Z3.Linq](../Z3-Linq2Z3/README.md) qui aborde la meme enigme via la couche declarative LINQ.
